# Phase 4 CJPE with c-3 (hourglass encoder-decoder): full fine-tuning

**Setup:** Add Input → Your Work → phase-1-c-3. Accelerator GPU T4 x2, Internet on.

Run order: Cell 0 → A → B → C → D → E0 → E1. To run E1 again from fresh Phase 1 weights, re-run Cell A and E0 first.

Your work in E1 (course rules): `[H]` hyperparameters are left empty (`None`) except your 1 epoch and 60-minute training cap; `[S]`, `[P]`, `[N]` are filled from your earlier full fine-tune and pooling code.

## Cell 0: classes from your Phase 1 c-3 notebook (cells 1, 8, 20, 22, 23, in that order)

In [1]:
# --- Phase 1 c-3 notebook, cell 1 (verbatim) ---
from collections import Counter
from tqdm.auto import tqdm
import pickle
import re
from collections import Counter, defaultdict

In [2]:
# --- Phase 1 c-3 notebook, cell 8 (verbatim) ---


SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]  # fixed ids 0-3, never merged

class BPETokenizer:
    def __init__(self):
        self.id_to_bytes = {}
        self.bytes_to_id = {}
        self.merges = {}
        self.merge_order = []
        self.merge_ranks = {}  # Precomputed pair -> rank for fast encoding
        self.cache = {}        # Chunk lookup cache
        self._vocab_size = 0
        
        # Splits on whitespace+word or just whitespace. 
        # Guarantees round-trip: "".join(chunks) == text
        self.split_pattern = re.compile(r'\s*\S+|\s+')

    def train(self, text: str, vocab_size: int):
        # 1. Setup base vocabulary
        for i, tok in enumerate(SPECIAL_TOKENS):
            b = tok.encode("utf-8")
            self.id_to_bytes[i] = b
            self.bytes_to_id[b] = i

        next_id = len(SPECIAL_TOKENS)
        byte_to_base_id = {}
        for value in range(256):
            b = bytes([value])
            self.id_to_bytes[next_id] = b
            self.bytes_to_id[b] = next_id
            byte_to_base_id[value] = next_id
            next_id += 1

        # 2. Split text into chunks and get frequencies
        chunks = self.split_pattern.findall(text)
        chunk_counts = Counter(chunks)
        
        # 3. Store each distinct chunk as a tuple of base byte ids
        chunk_to_ids = {
            chunk: tuple(byte_to_base_id[b] for b in chunk.encode("utf-8"))
            for chunk in chunk_counts
        }

        # 4. Initialize weighted pair counts and the pair-to-chunk index
        pair_counts = Counter()
        pair_to_chunks = defaultdict(set)
        
        def get_pairs(ids):
            return [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]

        for chunk, ids in chunk_to_ids.items():
            count = chunk_counts[chunk]
            for pair in get_pairs(ids):
                pair_counts[pair] += count
                pair_to_chunks[pair].add(chunk)

        target = vocab_size
        num_merges = target - next_id
        pbar = tqdm(total=num_merges, desc="Training BPE")

        # 5. Iteratively merge the best pair
        while next_id < target:
            if not pair_counts:
                break

            max_freq = max(pair_counts.values())
            if max_freq <= 1:
                break

            # Min tuple tie-breaker matching your original behavior
            best_pair = min(p for p, c in pair_counts.items() if c == max_freq)

            new_id = next_id
            self.merges[best_pair] = new_id
            self.merge_order.append(best_pair)
            self.merge_ranks[best_pair] = len(self.merge_order) - 1
            
            self.id_to_bytes[new_id] = self.id_to_bytes[best_pair[0]] + self.id_to_bytes[best_pair[1]]
            self.bytes_to_id[self.id_to_bytes[new_id]] = new_id

            # 6. Update only the chunks containing the merged pair
            chunks_to_update = list(pair_to_chunks[best_pair])
            for chunk in chunks_to_update:
                old_ids = chunk_to_ids[chunk]
                count = chunk_counts[chunk]
                
                # Subtract old pair combinations
                for pair in get_pairs(old_ids):
                    pair_counts[pair] -= count
                    if pair_counts[pair] <= 0:
                        del pair_counts[pair]
                    if chunk in pair_to_chunks[pair]:
                        pair_to_chunks[pair].remove(chunk)
                
                # Apply the merge inside this chunk
                new_ids = []
                i = 0
                while i < len(old_ids):
                    if i < len(old_ids) - 1 and (old_ids[i], old_ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(old_ids[i])
                        i += 1
                new_ids = tuple(new_ids)
                chunk_to_ids[chunk] = new_ids
                
                # Add new pair combinations created by the merge
                for pair in get_pairs(new_ids):
                    pair_counts[pair] += count
                    pair_to_chunks[pair].add(chunk)

            next_id += 1
            pbar.update(1)

        pbar.close()
        self._vocab_size = next_id

    def encode(self, text: str):
        chunks = self.split_pattern.findall(text)
        out_ids = []
        
        for chunk in chunks:
            # Check the dictionary cache first
            if chunk in self.cache:
                out_ids.extend(self.cache[chunk])
                continue
            
            # Cache miss: break into bytes and process
            chunk_bytes = chunk.encode("utf-8")
            ids = [self.bytes_to_id[bytes([b])] for b in chunk_bytes]
            
            # Merge until no adjacent pair has a valid merge rank
            while len(ids) > 1:
                pairs = [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]
                best_pair = None
                best_rank = float('inf')
                
                # Find the pair with the lowest rank
                for p in pairs:
                    rank = self.merge_ranks.get(p, float('inf'))
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = p
                
                if best_pair is None:
                    break  # No more valid merges for this chunk
                    
                # Apply the best merge to all occurrences in the chunk
                new_id = self.merges[best_pair]
                new_ids = []
                i = 0
                while i < len(ids):
                    if i < len(ids) - 1 and (ids[i], ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(ids[i])
                        i += 1
                ids = new_ids
            
            # Store result in cache and append
            self.cache[chunk] = ids
            out_ids.extend(ids)
            
        return out_ids

    def decode(self, ids):
        out = b"".join(self.id_to_bytes[i] for i in ids)
        return out.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return self._vocab_size

    @property
    def vocab_size(self):
        return self._vocab_size


def train_tokenizer(train_text: str, vocab_size: int):
    """Train the A1 BPE tokenizer on `train_text` and return it, ready for A3."""
    tok = BPETokenizer()
    tok.train(train_text, vocab_size)
    return tok

In [3]:
# --- Phase 1 c-3 notebook, cell 20 (verbatim) ---
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F

In [4]:
# --- Phase 1 c-3 notebook, cell 22 (verbatim) ---
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Seq2SeqConfig:
    vocab_size: int
    d_model: int = 512
    n_heads: int = 8
    d_ff: int = 2048
    max_seq_len: int = 512
    dropout: float = 0.1
    shorten_factor: int = 4
    n_pre_layers: int = 1
    n_bottleneck_layers: int = 4
    n_post_layers: int = 1

# ---------------------------------------------------------
# 1. Attention Modules
# ---------------------------------------------------------
class EncoderSelfAttention(nn.Module):
    """Bidirectional self-attention for the encoder (no causal mask)."""
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv_proj(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.n_heads, self.d_head).transpose(1, 2) for t in qkv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class CausalSelfAttention(nn.Module):
    """Causal self-attention for the decoder (looks only at past tokens)."""
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("mask", mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv_proj(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.n_heads, self.d_head).transpose(1, 2) for t in qkv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.mask[:T, :T], float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class CrossAttention(nn.Module):
    """Cross-attention: Query from Decoder, Key/Value from Encoder."""
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.kv_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        B_c, T_c, C_c = context.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        kv = self.kv_proj(context).chunk(2, dim=-1)
        k, v = [t.view(B_c, T_c, self.n_heads, self.d_head).transpose(1, 2) for t in kv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

# ---------------------------------------------------------
# 2. Feed-Forward & Standard Blocks
# ---------------------------------------------------------
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor):
        return self.net(x)


class EncoderBlock(nn.Module):
    """Standard bidirectional Transformer encoder block."""
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.self_attn = EncoderSelfAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.self_attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class DecoderBlock(nn.Module):
    """Standard causal Transformer decoder block with cross-attention."""
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.self_attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout)
        
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.cross_attn = CrossAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        
        self.ln3 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x: torch.Tensor, encoder_out: torch.Tensor) -> torch.Tensor:
        x = x + self.self_attn(self.ln1(x))
        x = x + self.cross_attn(self.ln2(x), encoder_out)
        x = x + self.ffn(self.ln3(x))
        return x

# ---------------------------------------------------------
# 3. Hierarchical Resampling Operations
# ---------------------------------------------------------
class Downsample(nn.Module):
    def __init__(self, factor: int):
        super().__init__()
        self.factor = factor

    def forward(self, x: torch.Tensor):
        B, L, D = x.shape
        x = x.view(B, L // self.factor, self.factor, D)
        return x.mean(dim=2)


class Upsample(nn.Module):
    def __init__(self, factor: int):
        super().__init__()
        self.factor = factor

    def forward(self, x: torch.Tensor):
        return x.repeat_interleave(self.factor, dim=1)


def shift_right(x: torch.Tensor, shift: int):
    if shift == 0:
        return x
    shifted = F.pad(x, (0, 0, shift, 0), value=0.0)
    return shifted[:, :-shift, :]

# ---------------------------------------------------------
# 4. Hourglass Blocks (Encoder & Decoder versions)
# ---------------------------------------------------------
class EncoderHourglassBlock(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig, factor: int, bottleneck_layers: int):
        super().__init__()
        self.factor = factor
        self.downsample = Downsample(factor)
        self.bottleneck = nn.ModuleList([EncoderBlock(cfg) for _ in range(bottleneck_layers)])
        self.upsample = Upsample(factor)

    def forward(self, x: torch.Tensor):
        residual = x
        shortened_x = self.downsample(x) # No shift for Encoder
        for block in self.bottleneck:
            shortened_x = block(shortened_x)
        expanded_x = self.upsample(shortened_x)
        return expanded_x + residual


class DecoderHourglassBlock(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig, factor: int, bottleneck_layers: int):
        super().__init__()
        self.factor = factor
        self.downsample = Downsample(factor)
        self.bottleneck = nn.ModuleList([DecoderBlock(cfg) for _ in range(bottleneck_layers)])
        self.upsample = Upsample(factor)

    def forward(self, x: torch.Tensor, encoder_out: torch.Tensor):
        residual = x
        x_shifted = shift_right(x, shift=self.factor - 1) # Shift required for Decoder causality
        shortened_x = self.downsample(x_shifted)
        for block in self.bottleneck:
            shortened_x = block(shortened_x, encoder_out)
        expanded_x = self.upsample(shortened_x)
        return expanded_x + residual

# ---------------------------------------------------------
# 5. Top-Level Hourglass Seq2Seq Model
# ---------------------------------------------------------
class HourglassEncoderDecoder(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        
        # Positional Encoding (using standard learned embedding for simplicity, 
        # replace with sinusoidal if strictly required)
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)

        # ENCODER
        self.encoder_pre = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_pre_layers)])
        self.encoder_hourglass = EncoderHourglassBlock(cfg, cfg.shorten_factor, cfg.n_bottleneck_layers)
        self.encoder_post = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_post_layers)])
        self.encoder_ln = nn.LayerNorm(cfg.d_model)

        # DECODER
        self.decoder_pre = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_pre_layers)])
        self.decoder_hourglass = DecoderHourglassBlock(cfg, cfg.shorten_factor, cfg.n_bottleneck_layers)
        self.decoder_post = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_post_layers)])
        self.decoder_ln = nn.LayerNorm(cfg.d_model)
        
        # Output Head
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight # Weight tying

    def get_positional_embeddings(self, seq_len: int, device: torch.device):
        positions = torch.arange(seq_len, device=device).unsqueeze(0)
        return self.pos_emb(positions)

    def forward(self, enc_idx: torch.Tensor, dec_idx: torch.Tensor) -> torch.Tensor:
        B_enc, T_enc = enc_idx.shape
        B_dec, T_dec = dec_idx.shape
        
        # 1. ENCODER PASS
        enc_x = self.tok_emb(enc_idx) + self.get_positional_embeddings(T_enc, enc_idx.device)
        for block in self.encoder_pre:
            enc_x = block(enc_x)
        enc_x = self.encoder_hourglass(enc_x)
        for block in self.encoder_post:
            enc_x = block(enc_x)
        encoder_out = self.encoder_ln(enc_x)

        # 2. DECODER PASS
        dec_x = self.tok_emb(dec_idx) + self.get_positional_embeddings(T_dec, dec_idx.device)
        for block in self.decoder_pre:
            dec_x = block(dec_x, encoder_out)
        dec_x = self.decoder_hourglass(dec_x, encoder_out)
        for block in self.decoder_post:
            dec_x = block(dec_x, encoder_out)
            
        dec_x = self.decoder_ln(dec_x)
        logits = self.lm_head(dec_x)
        
        return logits

In [5]:
# --- Phase 1 c-3 notebook, cell 23 (verbatim) ---
# ============================================================
# CELL 1 — Decoder-only Transformer (your A3 Q1 code, unchanged except the 2 marked LayerNorm lines)
# Replaces: §9 "Define The Model" (GPT2Config / GPT2LMHeadModel) and the DataCollator cell.
# ============================================================
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------- A3 Q1.1: manual causal multi-head self-attention ----------
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # None means the standard Transformer scaling 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.d_head)
        q = q.transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head)
        k = k.transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head)
        v = v.transpose(1, 2)

        scores = q @ k.transpose(-2, -1) * self.attention_scale
        mask = self.causal_mask[:T, :T]
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


# ---------- A3 Q1.2: decoder block and language model ----------
@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 560
    n_layers: int = 6
    n_heads: int = 8
    n_kv_heads: int = 4          # == n_heads for plain MHA; unused in this version
    d_ff: int = 2000
    max_seq_len: int = 256
    dropout: float = 0.15
    pos_emb_type: str = "sinusoidal"
    attention_scale: float = None
    norm_first: bool = False     # True = pre-norm, False = post-norm


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    pe = torch.zeros(max_len, d_model)
    for a in range(max_len):
        for b in range(d_model):
            i = b // 2
            angle = a / (10000 ** (2 * i / d_model))
            if b % 2 == 0:
                pe[a][b] = math.sin(angle)
            else:
                pe[a][b] = math.cos(angle)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout, attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm_first = cfg.norm_first

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))      # FIX: your A3 code used self.ln1 here (ln2 was never used)
        else:
            x = self.ln1(x + self.attn(x))     # FIX: your A3 code used self.ln2 here (ln1 was never used)
            x = self.ln2(x + self.ffn(x))
        return x


@dataclass
class EncDecConfig(Seq2SeqConfig):
    enc_len: int = None      # tokens the encoder reads (prompt side)
    dec_len: int = None      # tokens the decoder learns to produce (continuation side)


class EncDecLM(nn.Module):
    """Wrapper that keeps one forward(enc_idx, dec_idx) -> logits (B, T_dec, V) and pads the DECODER input on
    the right to a multiple of shorten_factor (safe: the decoder is causal, padding cannot change earlier logits).
    The ENCODER input is bidirectional, so it must already be a multiple of shorten_factor (never padded)."""
    def __init__(self, cfg: EncDecConfig):
        super().__init__()
        assert None not in (cfg.enc_len, cfg.dec_len), "set enc_len and dec_len in the config"
        self.cfg = cfg
        self.core = HourglassEncoderDecoder(cfg)          # YOUR model

    def forward(self, enc_idx, dec_idx):
        f = self.cfg.shorten_factor
        assert enc_idx.size(1) % f == 0, "encoder length must be a multiple of shorten_factor"
        T = dec_idx.size(1)
        pad = (-T) % f
        if pad:
            dec_idx = F.pad(dec_idx, (0, pad), value=0)    # id 0 = <pad>
        logits = self.core(enc_idx, dec_idx)
        return logits[:, :T, :]

In [6]:
# HF login (Phase 1 cell 2), needed if the dataset download asks for auth
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## Cell A: find and load the c-3 checkpoint and tokenizer

In [7]:
!find /kaggle/input -path "*phase-1-c-3*" \( -name "*.pt" -o -name "*.pkl" \) -exec ls -la {} \;

-rw-r--r-- 1 nobody nogroup 476977 Oct 21 13:53 /kaggle/input/notebooks/rudrajain2006/phase-1-c-3-hierarchial-encoder-decoder/cs781_phase1_full/bpe_tokenizer.pkl
-rw-r--r-- 1 nobody nogroup 476977 Oct 21 13:54 /kaggle/input/notebooks/rudrajain2006/phase-1-c-3-hierarchial-encoder-decoder/cs781_phase1_full/model_encdec/bpe_tokenizer.pkl
-rw-r--r-- 1 nobody nogroup 220615443 Oct 21 14:12 /kaggle/input/notebooks/rudrajain2006/phase-1-c-3-hierarchial-encoder-decoder/cs781_phase1_full/model_encdec/model_final.pt


In [8]:
# Expect <All keys matched successfully>, 55.13M params, vocab 11022, d_model 560, max_seq_len 1536, shorten 4
import torch, pickle, json, re, os
ROOT = "/kaggle/input/notebooks/rudrajain2006/phase-1-c-3-hierarchial-encoder-decoder/cs781_phase1_full"   # fix to what ls showed
device = torch.device("cuda")

tok = pickle.load(open(f"{ROOT}/model_encdec/bpe_tokenizer.pkl", "rb"))   # c-3's own tokenizer (vocab 11022)
ck = torch.load(f"{ROOT}/model_encdec/model_final.pt", map_location="cpu", weights_only=False)
cfg = EncDecConfig(**ck["config"])
lm = EncDecLM(cfg); print(lm.load_state_dict(ck["model"])); lm.to(device).eval()
del ck
print("params", sum(p.numel() for p in lm.parameters())/1e6, "M | vocab", tok.vocab_size,
      "| d_model", cfg.d_model, "| max_seq_len", cfg.max_seq_len, "| shorten", cfg.shorten_factor)

<All keys matched successfully>
params 55.13008 M | vocab 11022 | d_model 560 | max_seq_len 1536 | shorten 4


## Cell B: per-token features from the encoder (sanity check should print max diff 0.0)

In [9]:
def features(idx):                        # idx: (B, T) LongTensor; T must be a multiple of 4 and <= 1536
    c = lm.core
    T = idx.size(1)
    assert T % c.cfg.shorten_factor == 0 and T <= c.cfg.max_seq_len
    x = c.tok_emb(idx) + c.get_positional_embeddings(T, idx.device)
    for blk in c.encoder_pre:
        x = blk(x)
    x = c.encoder_hourglass(x)            # 4 bottleneck layers on the 4x shortened sequence, upsampled back
    for blk in c.encoder_post:
        x = blk(x)
    return c.encoder_ln(x)                # (B, T, 560)

# Sanity check: same as the encoder output inside the full model's forward
got = {}
h = lm.core.encoder_ln.register_forward_hook(lambda m, i, o: got.__setitem__("e", o))
idx = torch.randint(4, cfg.vocab_size, (1, 64), device=device)
with torch.no_grad():
    lm(idx, torch.full((1, 4), 2, device=device)); f = features(idx)
h.remove(); print("max diff", (f - got["e"]).abs().max().item(), tuple(f.shape))   # expect 0.0, (1, 64, 560)

max diff 0.0 (1, 64, 560)


## Cell C: Phase 4 CJPE data

In [10]:
# =====================================================================================
# CELL C — Phase 4 (CJPE) data: load train / dev / test cases and check the label field
# One case = one long judgment -> one binary label (0 / 1).
# =====================================================================================
import json
from collections import Counter
from huggingface_hub import hf_hub_download, list_repo_files

REPO = "Exploration-Lab/CS-781-Capstone"
print("Phase 4 files on HF:", [f for f in list_repo_files(REPO, repo_type="dataset") if "Phase_4" in f])

def get4(name):
    """Reads Phase_4_CJPE/<name>. Accepts plain JSON or JSON-lines."""
    p = hf_hub_download(REPO, f"Phase_4_CJPE/{name}", repo_type="dataset")
    with open(p, encoding="utf-8") as f:
        txt = f.read()
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        return [json.loads(line) for line in txt.splitlines() if line.strip()]

_raw = get4("Train/train.json")
_first = _raw[0] if isinstance(_raw, list) else next(iter(_raw.values()))
print("train layout:", type(_raw).__name__, len(_raw), "| first record keys:",
      list(_first) if isinstance(_first, dict) else type(_first).__name__)

TEXT_FIELDS  = ("text", "document", "case_text", "judgment", "sentences")
LABEL_FIELDS = ("label", "labels", "outcome", "decision", "target", "y", "pred")

def as_cases(raw, name):
    """-> {case_id: {"text": str, "label": 0/1 or None}}, in file order.
    Accepts a list of dicts, {case_id: {...}}, a one-key wrapper like {"data": [...]},
    or columns {"id": [...], "text": [...], ("label": [...])}. Other fields (expert_1..5) are ignored.
    Repeated ids (train merges the single- and multi-instance sets):
      same text and label as the first copy -> exact duplicate, dropped;
      different text or label              -> kept as a separate case "<id>#dup<n>" (train/dev),
                                              or dropped with a warning (test: one prediction per id)."""
    while isinstance(raw, dict) and len(raw) == 1 and isinstance(next(iter(raw.values())), (list, dict)):
        raw = next(iter(raw.values()))
    if isinstance(raw, dict) and "id" in raw and isinstance(raw["id"], list):
        keys = list(raw)
        raw = [{k: raw[k][i] for k in keys} for i in range(len(raw["id"]))]
    items = [(d.get("id", n), d) for n, d in enumerate(raw)] if isinstance(raw, list) else list(raw.items())
    cases, n_exact, n_diff, n_conflict = {}, 0, 0, 0
    for k, d in items:
        assert isinstance(d, dict), f"case {k}: unexpected entry {type(d).__name__}"
        k = str(d.get("id", k))
        tf = next((f for f in TEXT_FIELDS if f in d), None)
        assert tf is not None, f"case {k}: no text field (keys {list(d)})"
        text = d[tf]
        if isinstance(text, list):                    # a list of sentences/paragraphs
            text = " ".join(map(str, text))
        lf = next((f for f in LABEL_FIELDS if f in d), None)
        lab = d[lf] if lf is not None else None
        if isinstance(lab, list):
            assert len(lab) == 1, f"case {k}: label is a list {lab}"
            lab = lab[0]
        rec = {"text": str(text), "label": int(lab) if lab is not None else None}
        if k in cases:
            first = cases[k]
            if first["text"] == rec["text"] and first["label"] == rec["label"]:
                n_exact += 1
                continue
            if name == "test":
                print(f"WARNING test id {k} appears twice with different text: keeping the first")
                continue
            n_diff += 1
            n_conflict += first["text"] == rec["text"] and first["label"] != rec["label"]
            j = 1
            while f"{k}#dup{j}" in cases:
                j += 1
            k = f"{k}#dup{j}"
        cases[k] = rec
    print(f"{name}: {len(items)} records -> {len(cases)} cases | exact duplicates dropped {n_exact} | "
          f"repeated ids with different text/label kept {n_diff} (same text, different label: {n_conflict})")
    return cases

train = as_cases(_raw, "train")
dev   = as_cases(get4("Train/dev.json"), "dev")
test  = as_cases(get4("Test/test.json"), "test")
del _raw

for name, split in (("train", train), ("dev", dev), ("test", test)):
    labs = Counter(d["label"] for d in split.values())
    chars = sorted(len(d["text"]) for d in split.values())
    print(f"{name:5s}: {len(split):6d} cases | labels {dict(labs)} | chars median {chars[len(chars)//2]} max {chars[-1]}")
assert all(d["label"] in (0, 1) for s in (train, dev) for d in s.values()), "train/dev labels must be 0/1: check LABEL_FIELDS"
# The test file carries a label field too (placeholder or hidden): never use it, it is cleared here
_tl = Counter(d["label"] for d in test.values())
if any(v is not None for v in _tl):
    print(f"test label field values (ignored, set to None): {dict(_tl)}")
for d in test.values():
    d["label"] = None
print("first test ids:", list(test)[:3])
print("note: write_prediction uses the test ids above, so every test id gets exactly one entry")
print(f"train∩dev ids {len(set(train) & set(dev))} | train∩test {len(set(train) & set(test))}")

Phase 4 files on HF: ['Phase_4_CJPE/Test/test.json', 'Phase_4_CJPE/Train/dev.json', 'Phase_4_CJPE/Train/train.json']


Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

train layout: list 37387 | first record keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
train: 37387 records -> 32305 cases | exact duplicates dropped 5082 | repeated ids with different text/label kept 0 (same text, different label: 0)


Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

dev: 1988 records -> 994 cases | exact duplicates dropped 994 | repeated ids with different text/label kept 0 (same text, different label: 0)


Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

test: 1517 records -> 1517 cases | exact duplicates dropped 0 | repeated ids with different text/label kept 0 (same text, different label: 0)
train:  32305 cases | labels {0: 18920, 1: 13385} | chars median 12120 max 2492701
dev  :    994 cases | labels {0: 497, 1: 497} | chars median 16708 max 260198
test :   1517 cases | labels {1: 762, 0: 755} | chars median 17320 max 233000
test label field values (ignored, set to None): {1: 762, 0: 755}
first test ids: ['1986_397', '1977_183', '1993_98']
note: write_prediction uses the test ids above, so every test id gets exactly one entry
train∩dev ids 0 | train∩test 0


## Cell D: tokenise, scorer, writer, majority baseline

In [11]:
# =====================================================================================
# CELL D — tokenise every case once, length statistics, local scorer, prediction writer,
#          majority baseline (checks the scorer and writer end to end before any modelling)
# =====================================================================================
import os, json, pickle, zipfile
import numpy as np
from collections import Counter
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

# Tokenising long judgments with the Phase 1 BPE is slow, so the ids are cached in /kaggle/working.
TOK_CACHE = "/kaggle/working/c3_cjpe_ids.pkl"
_ids = pickle.load(open(TOK_CACHE, "rb")) if os.path.exists(TOK_CACHE) else None
if _ids is None or any(set(_ids.get(n, {})) != set(sp) for n, sp in (("train", train), ("dev", dev), ("test", test))):
    # no cache, or Cell C now gives different case ids: tokenise again
    _ids = {}
    for name, split in (("train", train), ("dev", dev), ("test", test)):
        _ids[name] = {k: tok.encode(d["text"]) for k, d in tqdm(split.items(), desc=f"tokenising {name}")}
    pickle.dump(_ids, open(TOK_CACHE, "wb"))
for name, split in (("train", train), ("dev", dev), ("test", test)):
    for k, d in split.items():
        d["ids"] = _ids[name][k]
del _ids

# Length statistics in c-3 tokens: these drive WINDOW / STRIDE / MAX_CHUNKS in Cell E1
# (c-3 reads at most 1536 tokens per row, and every row must be a multiple of 4)
for name, split in (("train", train), ("dev", dev), ("test", test)):
    n = np.array([len(d["ids"]) for d in split.values()])
    print(f"{name:5s} tokens: min {n.min()} median {np.median(n):.0f} p90 {np.percentile(n, 90):.0f} "
          f"max {n.max()} | > 1536 tokens: {100 * (n > 1536).mean():.1f}%")


# -------------------------------------------------------------------------------------
# Local scorer: macro-F1 over the two classes (the Codabench metric), plus P / R
# -------------------------------------------------------------------------------------
def evaluate(pred, cases, show=True):
    """pred: {case_id: 0/1}."""
    ids = list(cases)
    g = [cases[k]["label"] for k in ids]
    p = [pred[k] for k in ids]
    res = {"f1": f1_score(g, p, average="macro", labels=[0, 1], zero_division=0),
           "p": precision_score(g, p, average="macro", labels=[0, 1], zero_division=0),
           "r": recall_score(g, p, average="macro", labels=[0, 1], zero_division=0),
           "f1_per_class": f1_score(g, p, average=None, labels=[0, 1], zero_division=0).tolist(),
           "cm": confusion_matrix(g, p, labels=[0, 1])}
    if show:
        print(f"macro-F1 {res['f1']:.4f}  P {res['p']:.4f}  R {res['r']:.4f} | "
              f"F1 class0 {res['f1_per_class'][0]:.4f} class1 {res['f1_per_class'][1]:.4f}")
        print("confusion (rows gold 0/1, cols pred 0/1):\n", res["cm"])
    return res


# -------------------------------------------------------------------------------------
# Writer: prediction.json = a LIST of {"id", "pred"} (not a dict as in Phase 3), zipped
# -------------------------------------------------------------------------------------
def write_prediction(pred, test_cases, out_dir="/kaggle/working"):
    out = []
    for k in test_cases:
        v = pred[k]
        assert int(v) in (0, 1) and float(v) == int(v), f"{k}: pred {v!r} is not 0/1"
        out.append({"id": k, "pred": int(v)})                 # Python int (numpy ints break json.dump)
    assert len(out) == len(test_cases) == len({o["id"] for o in out}), "every test id once, no extras"
    assert set(pred) >= set(test_cases), "missing test ids"
    path = os.path.join(out_dir, "prediction.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f)
    zpath = os.path.join(out_dir, "submission.zip")
    with zipfile.ZipFile(zpath, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    with zipfile.ZipFile(zpath) as z:
        assert z.namelist() == ["prediction.json"], z.namelist()
    print(f"{len(out)} cases | pred counts {dict(Counter(o['pred'] for o in out))} -> {zpath}")


# -------------------------------------------------------------------------------------
# Majority baseline (format check only; don't spend a Codabench slot on it)
# -------------------------------------------------------------------------------------
majority = Counter(d["label"] for d in train.values()).most_common(1)[0][0]
print("majority class:", majority)
evaluate({k: majority for k in dev}, dev)
write_prediction({k: majority for k in test}, test)

tokenising train:   0%|          | 0/32305 [00:00<?, ?it/s]

tokenising dev:   0%|          | 0/994 [00:00<?, ?it/s]

tokenising test:   0%|          | 0/1517 [00:00<?, ?it/s]

train tokens: min 55 median 2692 p90 8584 max 541168 | > 1536 tokens: 69.1%
dev   tokens: min 217 median 4234 p90 10414 max 70685 | > 1536 tokens: 93.0%
test  tokens: min 107 median 4416 p90 10912 max 64341 | > 1536 tokens: 92.3%
majority class: 0
macro-F1 0.3333  P 0.2500  R 0.5000 | F1 class0 0.6667 class1 0.0000
confusion (rows gold 0/1, cols pred 0/1):
 [[497   0]
 [497   0]]
1517 cases | pred counts {0: 1517} -> /kaggle/working/submission.zip


## Cell E0: shared plumbing (c-3 encoder path, chunks, training loop with time cap, submission)

In [12]:
# =====================================================================================
# CELL E0 — shared plumbing for the c-3 Phase 4 (CJPE) experiments
# Run once after Cells 0, A, B, C, D. Then run the technique cell (E1 full fine-tuning).
# Needs: lm, cfg, device, features, SPECIAL_TOKENS, train, dev, test (with d["ids"]),
#        evaluate, write_prediction
#
# Nothing in this cell is a model design or a hyperparameter choice. It only does:
# the c-3 encoder path, cutting a long judgment into encoder rows (one or more chunks per
# case), batching, the training loop, prediction, error listing and packaging.
# Lines marked [c-3] are what the hourglass encoder needs (rows a multiple of 4, no mask).
# =====================================================================================
import math, time, random
from collections import Counter
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm

N_CLASSES = 2
FEAT_DIM = cfg.d_model    # 560
SF = cfg.shorten_factor   # [c-3] 4: every encoder row length must be a multiple of this
PAD_ID = SPECIAL_TOKENS.index("<pad>")
UNK_ID = SPECIAL_TOKENS.index("<unk>")
GRAD_CHECKPOINT = False   # memory switch, set in E1 (same maths, slower; recomputes blocks in backward)

# The decoder is never used for CJPE; dropping it frees GPU memory. tok_emb stays (shared with lm_head).
lm.core.decoder_pre = None          # [c-3] the hourglass decoder has 4 parts
lm.core.decoder_hourglass = None
lm.core.decoder_post = None
lm.core.decoder_ln = None
torch.cuda.empty_cache()


# -------------------------------------------------------------------------------------
# [c-3] encoder path (same maths as Cell B features(), on any backbone object).
# The hourglass forward is written out so each block can be gradient-checkpointed.
# -------------------------------------------------------------------------------------
def _run(blk, x):
    if GRAD_CHECKPOINT and torch.is_grad_enabled():
        return checkpoint(blk, x, use_reentrant=False)
    return blk(x)

def encoder_forward(backbone, idx):
    c = backbone.core
    T = idx.size(1)
    assert T <= c.cfg.max_seq_len, "row longer than 1536"
    assert T % c.cfg.shorten_factor == 0, "row length must be a multiple of 4"
    x = c.tok_emb(idx) + c.get_positional_embeddings(T, idx.device)
    for blk in c.encoder_pre:                   # 2 full-resolution blocks
        x = _run(blk, x)
    hg = c.encoder_hourglass                    # downsample x4 -> 4 bottleneck blocks -> upsample + residual
    s = hg.downsample(x)
    for blk in hg.bottleneck:
        s = _run(blk, s)
    x = hg.upsample(s) + x
    for blk in c.encoder_post:                  # 2 full-resolution blocks
        x = _run(blk, x)
    return c.encoder_ln(x)                      # (R, T, 560)

with torch.no_grad():                           # same output as Cell B
    _p = torch.randint(4, cfg.vocab_size, (1, 64), device=device)
    print("encoder_forward vs features max diff", (encoder_forward(lm, _p) - features(_p)).abs().max().item())


# -------------------------------------------------------------------------------------
# [c-3] Rows without padding (as in Phase 2/3 c-3): n tokens with n % 4 == 0 -> one row;
# otherwise two overlapping rows of L = 4*floor(n/4) (start and end), stitched back so each
# token is read once. Only n < 4 gets <pad> (never pooled).
# -------------------------------------------------------------------------------------
def rows_for(ids):
    n = len(ids)
    L = (n // SF) * SF
    if L == n:
        return [list(ids)], lambda s, e: [(0, s, e)]
    if L == 0:
        return [list(ids) + [PAD_ID] * (SF - n)], lambda s, e: [(0, s, e)]
    o = n - L                                   # row 1 starts at token o
    def seg(s, e):
        return [p for p in ((0, s, min(e, L)), (1, max(s, L) - o, e - o)) if p[2] > p[1]]
    return [list(ids[:L]), list(ids[o:])], seg


def make_windows(n, window, stride):
    starts = list(range(0, n - window, stride)) + [n - window]
    return [(s, s + window) for s in starts]


# -------------------------------------------------------------------------------------
# Document representation: which token spans of a case the encoder reads.
#   "head"      first `window` tokens
#   "tail"      last `window` tokens (judgments often end with the reasoning / order)
#   "head_tail" both of the above as two chunks
#   "chunks"    windows every `stride` tokens over the whole case; if there are more than
#               max_chunks, max_chunks evenly spaced ones are kept (None = keep all)
# A case of <= window tokens is always one chunk (the whole case).
# -------------------------------------------------------------------------------------
DOC_REPS = ("head", "tail", "head_tail", "chunks")

def doc_spans(n, rep, window, stride=None, max_chunks=None):
    if n <= window:
        return [(0, n)]
    if rep == "head":
        return [(0, window)]
    if rep == "tail":
        return [(n - window, n)]
    if rep == "head_tail":
        return [(0, window), (n - window, n)]
    assert rep == "chunks", rep
    wins = make_windows(n, window, stride)
    if max_chunks is not None and len(wins) > max_chunks:
        keep = sorted(set(np.linspace(0, len(wins) - 1, max_chunks).round().astype(int).tolist()))
        wins = [wins[i] for i in keep]
    return wins


def build_examples(cases, rep, window, stride=None, max_chunks=None):
    """One example per case: {"id", "rows": [[ids], ...], "chunks": [[(row, s, e), ...] per chunk], "label"}.
    A chunk's pieces concatenated are that chunk's tokens in order (pad-free stitching above)."""
    assert rep in DOC_REPS, rep
    assert window % SF == 0 and window <= cfg.max_seq_len, "[c-3] WINDOW: a multiple of 4, <= 1536"
    if rep == "chunks":
        assert stride is not None and 0 < stride <= window, "chunks: set STRIDE (<= WINDOW)"
    examples = []
    for k, d in tqdm(cases.items(), desc=f"examples ({rep})", leave=False):
        ids = d["ids"] if d["ids"] else [UNK_ID]
        rows, chunks = [], []
        for s, e in doc_spans(len(ids), rep, window, stride, max_chunks):
            r, seg = rows_for(ids[s:e])
            chunks.append([(len(rows) + ri, a, b) for ri, a, b in seg(0, e - s)])
            rows += r
        examples.append({"id": k, "rows": rows, "chunks": chunks,
                         "label": d["label"] if d["label"] is not None else -100})
    return examples


def make_batches(examples, batch_size, shuffle):
    """batch_size counts CASES (each case brings all its chunks)."""
    exs = list(examples)
    if shuffle:
        random.shuffle(exs)
    return [exs[i:i + batch_size] for i in range(0, len(exs), batch_size)]


def to_tensors(batch):
    """Rows of equal length go through the encoder together (no padding: the encoder has no mask)."""
    by_len, where, docs = {}, [], []
    for ex in batch:
        base = len(where)
        for r in ex["rows"]:
            g = by_len.setdefault(len(r), [])
            where.append((len(r), len(g))); g.append(r)
        docs.append([[(base + r, s, e) for r, s, e in ch] for ch in ex["chunks"]])
    return {"groups": {L: torch.tensor(rs, device=device) for L, rs in by_len.items()},
            "where": where, "docs": docs,
            "labels": torch.tensor([ex["label"] for ex in batch], device=device)}


class DocModel(nn.Module):
    """c-3 encoder -> token vectors -> pool_tokens per chunk -> pool_chunks per case -> head -> 2 logits.
    pool_tokens: your function/module, (L, 560) tokens of ONE chunk -> one vector.
    pool_chunks: your function/module, (C, d) chunk vectors of ONE case -> one vector.
    head: your module, (B, d) -> (B, 2)."""
    def __init__(self, backbone, pool_tokens, pool_chunks, head):
        super().__init__()
        self.backbone, self.pool_tokens, self.pool_chunks, self.head = backbone, pool_tokens, pool_chunks, head
    def forward(self, b):
        h = {L: encoder_forward(self.backbone, t) for L, t in b["groups"].items()}
        row = lambda r: h[b["where"][r][0]][b["where"][r][1]]
        vecs = []
        for chunks in b["docs"]:
            cv = torch.stack([self.pool_tokens(torch.cat([row(r)[s:e] for r, s, e in ch])) for ch in chunks])
            vecs.append(self.pool_chunks(cv))
        return self.head(torch.stack(vecs))                                  # (B, 2)


def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    print(f"trainable {tr/1e6:.3f}M / total {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot


# -------------------------------------------------------------------------------------
# Loss and class-balance helper
# -------------------------------------------------------------------------------------
def ce_loss(logits, labels, class_weights=None):
    return F.cross_entropy(logits.float(), labels, weight=class_weights, ignore_index=-100)

def inverse_freq_weights(cases):
    """Optional helper: n / (2 * count_c) per class, for CLASS_WEIGHTS if you choose to use it."""
    c = Counter(d["label"] for d in cases.values())
    n = sum(c.values())
    return torch.tensor([n / (2 * c[0]), n / (2 * c[1])], dtype=torch.float, device=device)


# -------------------------------------------------------------------------------------
# Prediction: {case_id: 0/1} and {case_id: P(class 1)}.
# threshold=None -> argmax; a number -> pred 1 when P(1) >= threshold.
# -------------------------------------------------------------------------------------
@torch.no_grad()
def predict(model, examples, batch_size, threshold=None):
    model.eval()
    pred, prob = {}, {}
    for batch in tqdm(make_batches(examples, batch_size, shuffle=False), desc="predict", leave=False):
        b = to_tensors(batch)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(b).float()
        p1 = logits.softmax(-1)[:, 1].tolist()
        am = logits.argmax(-1).tolist()
        for ex, p, a in zip(batch, p1, am):
            prob[ex["id"]] = p
            pred[ex["id"]] = int(a) if threshold is None else int(p >= threshold)
    assert len(pred) == len(examples), "every case gets a prediction"
    return pred, prob


def threshold_sweep(prob, cases, grid=None):
    """Optional post-processing check: dev macro-F1 for each threshold on P(1).
    Picking the threshold on dev makes the dev score optimistic; say so in the report."""
    grid = np.linspace(0.05, 0.95, 19) if grid is None else grid
    rows = [(float(t), evaluate({k: int(prob[k] >= t) for k in cases}, cases, show=False)["f1"]) for t in grid]
    for t, f in rows:
        print(f"  threshold {t:.2f}  dev macro-F1 {f:.4f}")
    return max(rows, key=lambda x: x[1])


# -------------------------------------------------------------------------------------
# Training loop. loss_fn(logits, batch_tensors) comes from the technique cell.
# Optional separate head LR; accum_steps = optimizer step every N batches (memory switch).
# Keeps the epoch with the best dev macro-F1 (argmax predictions).
# -------------------------------------------------------------------------------------
def steps_per_epoch(n_examples, batch_size, accum_steps=1):
    """For building an LR_LAMBDA schedule: optimizer steps in one epoch."""
    return math.ceil(math.ceil(n_examples / batch_size) / accum_steps)

def train_model(model, train_ex, dev_ex, dev_cases, loss_fn, *, lr, epochs, batch_size, head_lr=None,
                weight_decay=0.0, grad_clip=1.0, lr_lambda=None, accum_steps=1, max_train_minutes=None,
                save_path="best_trainable.pt"):
    """max_train_minutes: hard cap on time spent in training batches (dev scoring not counted).
    When it is reached, the current epoch stops after the batch in progress, the pending
    gradients are applied, dev is scored, the weights are saved, and training ends."""
    params = [p for p in model.parameters() if p.requires_grad]
    head_ids = {id(p) for p in model.head.parameters()}
    groups = [{"params": [p for p in params if id(p) not in head_ids], "lr": lr},
              {"params": [p for p in params if id(p) in head_ids], "lr": head_lr if head_lr is not None else lr}]
    opt = torch.optim.AdamW([g for g in groups if g["params"]], weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda) if lr_lambda else None
    scaler = torch.amp.GradScaler("cuda")
    torch.cuda.reset_peak_memory_stats()
    t0, best_f1, best_res, log, skipped = time.time(), -1.0, None, [], 0
    cap_s = max_train_minutes * 60 if max_train_minutes is not None else None
    used_s, capped = 0.0, False
    def opt_step():
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, grad_clip)
        scaler.step(opt); scaler.update()
        opt.zero_grad(set_to_none=True)
        if sched: sched.step()
    for epoch in range(epochs):
        model.train(); tot, nb, seen, pending = 0.0, 0, 0, False
        opt.zero_grad(set_to_none=True)
        batches = make_batches(train_ex, batch_size, shuffle=True)
        pbar = tqdm(batches, desc=f"epoch {epoch + 1}/{epochs}")
        te = time.time()
        for i, batch in enumerate(pbar):
            if cap_s is not None and used_s + (time.time() - te) >= cap_s:
                capped = True
                break
            seen += 1
            b = to_tensors(batch)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(b)
            loss = loss_fn(logits.float(), b)
            if not torch.isfinite(loss):
                skipped += 1
                continue
            scaler.scale(loss / accum_steps).backward()
            tot += loss.item(); nb += 1; pending = True
            if (i + 1) % accum_steps == 0 or i + 1 == len(batches):
                opt_step(); pending = False
            pbar.set_postfix(loss=f"{tot / max(1, nb):.4f}", skipped=skipped)
        if pending:                                   # stopped by the time cap mid-accumulation
            opt_step()
        pbar.close()
        used_s += time.time() - te
        if capped:
            print(f"time cap {max_train_minutes} min reached in epoch {epoch + 1}: trained on {seen}/{len(batches)} "
                  f"batches (~{min(seen * batch_size, len(train_ex))}/{len(train_ex)} cases)")
        res = evaluate(predict(model, dev_ex, batch_size)[0], dev_cases, show=False)
        log.append({"epoch": epoch + 1, "train_loss": tot / max(1, nb), "dev_f1": res["f1"],
                    "dev_p": res["p"], "dev_r": res["r"], "skipped": skipped,
                    "batches_seen": seen, "batches_total": len(batches), "train_min_so_far": round(used_s / 60, 1)})
        print(log[-1])
        if res["f1"] > best_f1:
            best_f1, best_res = res["f1"], res
            torch.save({n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}, save_path)
        if capped:
            break
    train_min = used_s / 60
    peak_gb = torch.cuda.max_memory_allocated() / 2**30
    print(f"best dev macro-F1 {best_f1:.4f} (P {best_res['p']:.4f}, R {best_res['r']:.4f}) | "
          f"train time {train_min:.1f} min (training batches only{', time-capped' if capped else ''}) | "
          f"wall time incl. dev scoring {(time.time() - t0) / 60:.1f} min | peak GPU mem {peak_gb:.2f} GB")
    return log, best_res, {"train_min": train_min, "peak_gb": peak_gb, "capped": capped}


@torch.no_grad()
def time_inference(model, examples, batch_size):
    """Wall-clock seconds to predict a set (for the efficiency table in the report)."""
    torch.cuda.synchronize(); t = time.time()
    predict(model, examples, batch_size)
    torch.cuda.synchronize(); return time.time() - t


def show_errors(pred, prob, cases, n=5, chars=300):
    """False positives / false negatives with P(1), for the error analysis in the report."""
    fp = sorted([k for k in cases if pred[k] == 1 and cases[k]["label"] == 0], key=lambda k: -prob[k])
    fn = sorted([k for k in cases if pred[k] == 0 and cases[k]["label"] == 1], key=lambda k: prob[k])
    for name, ks in (("FALSE POSITIVES (gold 0, pred 1)", fp), ("FALSE NEGATIVES (gold 1, pred 0)", fn)):
        print(f"--- {name}: {len(ks)} cases, most confident first")
        for k in ks[:n]:
            print(f"  {k}  P(1)={prob[k]:.3f}  tokens={len(cases[k]['ids'])}  | {cases[k]['text'][:chars]!r}")
    return fp, fn


def write_submission(model, build_kwargs, batch_size, best_path=None, threshold=None, out_dir="/kaggle/working"):
    """Reload the best trainable weights, predict test, write prediction.json + submission.zip."""
    if best_path:
        model.load_state_dict(torch.load(best_path), strict=False)
    test_ex = build_examples(test, **build_kwargs)
    pred, _ = predict(model, test_ex, batch_size, threshold)
    write_prediction(pred, test, out_dir=out_dir)

encoder_forward vs features max diff 0.0


## Cell E1: full fine-tuning (fill [H]; 1 epoch, 60-minute cap)

In [ ]:

# =====================================================================================
# CELL E1 — Full fine-tuning of the c-3 hourglass encoder for CJPE
# =====================================================================================

# -------------------------------------------------------------------------------------
# [H] Memory-conscious configuration for a 14.6 GiB GPU
# -------------------------------------------------------------------------------------
DOC_REP           = "chunks"
WINDOW            = 200
STRIDE            = 100
MAX_CHUNKS        = 8

LR                = 1e-5
HEAD_LR           = None
EPOCHS            =2
MAX_TRAIN_MINUTES = 60

BATCH_SIZE        = 1
WEIGHT_DECAY      = 0.0
LR_LAMBDA         = None
CLASS_WEIGHTS     = None
THRESHOLD         = None

GRAD_CHECKPOINT   = True
ACCUM_STEPS       = 4
SEED              = 42

for _n in ["DOC_REP", "WINDOW", "LR", "EPOCHS", "BATCH_SIZE", "WEIGHT_DECAY"]:
    assert globals()[_n] is not None, f"set {_n}"

assert DOC_REP in DOC_REPS, DOC_REPS
assert WINDOW % SF == 0 and WINDOW <= cfg.max_seq_len, \
    "[c-3] WINDOW must be a multiple of 4, <= 1536"

if DOC_REP == "chunks":
    assert STRIDE is not None and 0 < STRIDE <= WINDOW, "set STRIDE for chunks"

BUILD = dict(
    rep=DOC_REP,
    window=WINDOW,
    stride=STRIDE,
    max_chunks=MAX_CHUNKS
)

random.seed(SEED)
torch.manual_seed(SEED)

RUN_NAME = f"c3_fullft_{DOC_REP}_w{WINDOW}"


# -------------------------------------------------------------------------------------
# [S] Train the full c-3 encoder
# -------------------------------------------------------------------------------------
for p in lm.parameters():
    p.requires_grad = False

for p in lm.core.tok_emb.parameters():
    p.requires_grad = True

for p in lm.core.pos_emb.parameters():
    p.requires_grad = True

for p in lm.core.encoder_pre.parameters():
    p.requires_grad = True

for p in lm.core.encoder_hourglass.parameters():
    p.requires_grad = True

for p in lm.core.encoder_post.parameters():
    p.requires_grad = True

for p in lm.core.encoder_ln.parameters():
    p.requires_grad = True


# -------------------------------------------------------------------------------------
# [P] Pooling
# -------------------------------------------------------------------------------------
def pool_tokens(h):
    # (L, 560) -> (560,)
    return h.mean(0)

def pool_chunks(c):
    # (C, 560) -> (560,)
    return c.mean(0)


# -------------------------------------------------------------------------------------
# [N] Classification head: single linear layer
# -------------------------------------------------------------------------------------
head = nn.Linear(FEAT_DIM, N_CLASSES)

model = DocModel(
    lm,
    pool_tokens,
    pool_chunks,
    head
).to(device)

n_train_params, n_total_params = count_params(model)

assert any(p.requires_grad for p in lm.parameters()), \
    "no encoder parameter is trainable: set [S]"


# -------------------------------------------------------------------------------------
# Data
# -------------------------------------------------------------------------------------
train_ex = build_examples(train, **BUILD)
dev_ex = build_examples(dev, **BUILD)

nch = [len(ex["chunks"]) for ex in train_ex]

print(
    f"train cases {len(train_ex)} | dev cases {len(dev_ex)} | "
    f"chunks per case: mean {np.mean(nch):.1f} max {max(nch)} | "
    f"encoder rows per train epoch {sum(len(ex['rows']) for ex in train_ex)}"
)


# -------------------------------------------------------------------------------------
# Training
# -------------------------------------------------------------------------------------
log, best_res, cost = train_model(
    model,
    train_ex,
    dev_ex,
    dev,
    loss_fn=lambda logits, b: ce_loss(
        logits, b["labels"], CLASS_WEIGHTS
    ),
    lr=LR,
    head_lr=HEAD_LR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    weight_decay=WEIGHT_DECAY,
    lr_lambda=LR_LAMBDA,
    accum_steps=ACCUM_STEPS,
    max_train_minutes=MAX_TRAIN_MINUTES,
    save_path=f"/kaggle/working/{RUN_NAME}_best_trainable.pt"
)


# -------------------------------------------------------------------------------------
# Evaluation with best checkpoint
# -------------------------------------------------------------------------------------
BEST_PATH = f"/kaggle/working/{RUN_NAME}_best_trainable.pt"

state = torch.load(BEST_PATH, map_location=device, weights_only=True)
load_result = model.load_state_dict(state, strict=False)

print("Missing checkpoint keys:", load_result.missing_keys)
print("Unexpected checkpoint keys:", load_result.unexpected_keys)

dev_pred, dev_prob = predict(
    model,
    dev_ex,
    BATCH_SIZE,
    THRESHOLD
)

dev_res = evaluate(dev_pred, dev)
fp_ids, fn_ids = show_errors(dev_pred, dev_prob, dev)

infer_s = time_inference(model, dev_ex, BATCH_SIZE)

print({
    "run": RUN_NAME,
    "technique": "full FT",
    "rep": DOC_REP,
    "window": WINDOW,
    "stride": STRIDE,
    "max_chunks": MAX_CHUNKS,
    "trainable_M": round(n_train_params / 1e6, 3),
    "total_M": round(n_total_params / 1e6, 2),
    "peak_gb": round(cost["peak_gb"], 2),
    "train_min": round(cost["train_min"], 1),
    "time_capped": cost["capped"],
    "batches_seen": f'{log[-1]["batches_seen"]}/{log[-1]["batches_total"]}',
    "dev_infer_s": round(infer_s, 1),
    "dev_f1": round(dev_res["f1"], 4),
    "dev_p": round(dev_res["p"], 4),
    "dev_r": round(dev_res["r"], 4),
    "threshold": THRESHOLD
})


# -------------------------------------------------------------------------------------
# Test prediction and submission
# -------------------------------------------------------------------------------------
write_submission(
    model,
    BUILD,
    BATCH_SIZE,
    best_path=BEST_PATH,
    threshold=THRESHOLD
)


trainable 26.060M / total 26.06M (100.00%)


examples (chunks):   0%|          | 0/32305 [00:00<?, ?it/s]

examples (chunks):   0%|          | 0/994 [00:00<?, ?it/s]

train cases 32305 | dev cases 994 | chunks per case: mean 7.3 max 8 | encoder rows per train epoch 235350


epoch 1/1:   0%|          | 0/32305 [00:00<?, ?it/s]